# Open Darts - train the dart detector (free Colab GPU)

This notebook trains a small neural network that finds the **dart tips** and the **four calibration points** of the board
in a camera picture, at any camera angle. The app uses those four points to work out where the board is, so no manual
calibration is needed.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Download `cropped_images.zip` (the 800x800 version) from https://ieee-dataport.org/open-access/deepdarts-dataset
   (free IEEE account needed) and put it in your Google Drive, in a folder called `darts`.
3. Run the cells in order. Training takes roughly 1-3 hours. The last cell downloads `dartnet.tflite`.
4. Copy `dartnet.tflite` to `app/src/main/assets/dartnet.tflite` in the project and rebuild the app.

In [ ]:
!pip -q install ultralytics pandas
!git clone https://github.com/Yaswanth1826/open-darts.git
!wget -q -O labels.pkl https://github.com/wmcnally/deep-darts/raw/master/dataset/labels.pkl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p dataset && unzip -q -o /content/drive/MyDrive/darts/cropped_images.zip -d dataset
!cp labels.pkl dataset/
!find dataset -maxdepth 3 -type d | head

If the folder listing above does not show `dataset/cropped_images/800/...`, move the folders so that path exists, then continue.

In [ ]:
!python open-darts/training/prepare_deepdarts.py --dataset-dir dataset --out yolo_darts --oblique 2

In [ ]:
# Side-view photos with clear-flight darts (dartscribe, CC-BY-SA-4.0), about 2 GB; the corners and dart tips come from its labels.
!pip -q install pyyaml
!python open-darts/training/prepare_dartscribe.py --download dartscribe
!python open-darts/training/prepare_dartscribe.py --dataset-dir dartscribe --out yolo_darts

In [ ]:
!python open-darts/training/train.py --data yolo_darts/data.yaml --epochs 60 --imgsz 640 --batch 32

In [ ]:
from google.colab import files
files.download('dartnet.tflite')

**Second stage: the tip refiner (optional, recommended).** The detector sometimes marks a dart a few millimetres up the barrel. This small network looks at a close-up of each mark and moves it onto the real tip (and drops marks that are no tip). It trains in a few minutes on the pictures prepared above. Put the downloaded `tiprefine.tflite` into `app/src/main/assets/` next to `dartnet.tflite`; the app uses it when the file is there.


In [ ]:
!python open-darts/training/make_tip_patches.py --data yolo_darts --out tip_patches
!python open-darts/training/train_tip.py --patches tip_patches --out tiprefine.tflite

In [ ]:
from google.colab import files
files.download('tiprefine.tflite')

**Fine-tune on your own pictures (this is what makes it work on your board):**
1. Keep "Collect training photos" on in the app's camera menu and play. The photos land in *Pictures/OpenDarts/Training* on the phone.
2. Copy that folder to the PC and run `python training/label_photos.py Training` to fix the dart tips, then
   `python training/from_captures.py Training --out my_photos`.
3. Zip the `my_photos` folder (so it contains `images/` and `labels/`), put the zip in Drive at `darts/my_photos.zip`, and run the cells below.
A few hundred checked pictures of your own board and room make the model noticeably better.

In [ ]:
!mkdir -p my_photos && unzip -q -o /content/drive/MyDrive/darts/my_photos.zip -d my_photos
!python open-darts/training/prepare_deepdarts.py --dataset-dir dataset --out yolo_darts --oblique 2 --extra my_photos --extra-repeat 20

In [ ]:
# continues from the earlier weights; the val/test numbers now include a held-out part of your own photos
!python open-darts/training/train.py --data yolo_darts/data.yaml --model /content/drive/MyDrive/dartnet_runs/dartnet/weights/best.pt --epochs 25 --lr0 0.002 --project /content/drive/MyDrive/dartnet_runs2 --out dartnet_new.tflite

In [ ]:
from google.colab import files
files.download('dartnet_new.tflite')